# Chapter 2: Simulation & Data

This notebook walks through the SO-100 pick-and-place simulator, a scripted
baseline policy, and the LeRobot SO-101 expert dataset that feeds the
normalized DataLoader Chapter 3 imports. The arm family is SO-100 in sim,
SO-101 on hardware; the small kinematic gap is a Chapter 9 topic. Each
section maps cell-by-cell to a section of the book chapter; listings are
reproduced verbatim from the chapter plan.

**Run locally:** `pip install -e ".[dev,data,sim]"` from the repo root, then
open this notebook in Jupyter.

**Run on Colab:** the cell below detects Colab and installs the Vulkan ICDs
ManiSkill needs before doing the pip install.

In [ ]:
# Colab-only setup: install Vulkan ICDs + the chapter package.
# On a local machine with the package already installed, this cell is a no-op.
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !mkdir -p /usr/share/vulkan/icd.d
    !wget -q https://raw.githubusercontent.com/haosulab/ManiSkill/main/docker/nvidia_icd.json
    !wget -q https://raw.githubusercontent.com/haosulab/ManiSkill/main/docker/10_nvidia.json
    !mv nvidia_icd.json /usr/share/vulkan/icd.d
    !mv 10_nvidia.json /usr/share/glvnd/egl_vendor.d/10_nvidia.json
    !apt-get install -y --no-install-recommends libvulkan-dev
    !pip install -q "lrm-ch02[data,sim] @ git+https://github.com/Large-Robotics-Models-From-Scratch/lrm-code-chapter-2.git@main"


## 2.1 The SO-100 Pick-and-Place Environment

We use ManiSkill3's `PickCubeSO100-v1` task. The arm is SO-100 (close enough
to SO-101 that the kinematic gap is a Chapter 9 topic), the cube spawns in a
fixed region, and the goal is to drop the cube inside a target zone.

### Listing 2.1: Installing the SO-100 sim and creating the environment

Constructs the env with image observations and the default joint-space
control mode. The action space is `Box(6,)` — one delta per SO-100 joint.

In [ ]:
# pip install lerobot mani-skill
import gymnasium as gym
import mani_skill.envs
import matplotlib.pyplot as plt

env = gym.make(
    "PickCubeSO100-v1",
    obs_mode="rgb",
    control_mode="pd_joint_delta_pos",
    render_mode="rgb_array",
)
obs, info = env.reset(seed=42)
print(f"Observation keys: {list(obs.keys())}")
print(f"Action space: {env.action_space}")

frame = env.render()[0].cpu().numpy()
plt.imshow(frame); plt.axis("off"); plt.show()


### Listing 2.2: Running a random agent on PickCubeSO100-v1

`run_random_agent` samples uniformly from the action space across `n_episodes`
and returns `(success_rate, mean_return)`. Random actions almost never solve
pick-and-place — this is the performance floor every learned policy must
clear.

In [ ]:
import numpy as np
from ch02.viz import record_env_video

def run_random_agent(env, n_episodes=10):
    successes, returns = 0, []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=ep)
        ep_return = 0.0
        done = False
        while not done:
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            ep_return += reward
            done = terminated or truncated
        successes += int(info.get("success", False))
        returns.append(ep_return)
    return successes / n_episodes, np.mean(returns)

success_rate, mean_return = run_random_agent(env)
print(f"Random agent: success={success_rate:.0%} "
      f"return={mean_return:.2f}")

record_env_video(env, n_steps=200, seed=0, fps=20)


## 2.2 A Scripted Policy

Seven phases — approach, descend, grasp, lift, transport, place, release —
expressed as keyframe poses. ManiSkill's `SO100ArmMotionPlanningSolver`
solves IK and steps the joint trajectory for each keyframe.

### Listing 2.3: A multi-phase scripted pick-and-place policy

In [ ]:
import numpy as np
import sapien
from ch02.viz import plot_phase_keyframes

def run_scripted_episode(
        planner, grasp_pose, goal_pos):
    """Seven phases via the IK planner.

    approach → descend → grasp → lift → transport → place → hold.
    The final phase holds the cube at the goal instead of releasing —
    dropping it tends to bounce past the 1.56 cm success threshold.
    """
    quat = grasp_pose.q
    goal = np.asarray(goal_pos)

    planner.move_to_pose_with_screw(
        sapien.Pose([0, 0, 0.10]) * grasp_pose)
    planner.move_to_pose_with_screw(
        sapien.Pose([0, 0, 0.03]) * grasp_pose)
    planner.move_to_pose_with_screw(
        sapien.Pose([0, 0, 0.01]) * grasp_pose)
    planner.close_gripper(gripper_state=-0.8)
    planner.move_to_pose_with_screw(
        sapien.Pose([0, 0, 0.15]) * grasp_pose)
    planner.move_to_pose_with_screw(
        sapien.Pose(goal + np.array([0, 0, 0.05]), quat))
    planner.move_to_pose_with_screw(
        sapien.Pose(goal, quat))
    planner.hold(n_steps=30)

sample_grasp = sapien.Pose([0.0, 0.0, 0.05])
sample_goal = np.array([-0.2, 0.2, 0.02])
plot_phase_keyframes(sample_grasp, sample_goal)


### Listing 2.4: Evaluating the scripted policy

`run_scripted_agent` (from `ch02.scripted`) handles motion-planner setup,
grasp-pose computation, and success counting. We pass a state-mode env
since the policy reads cube/goal positions directly from `env.unwrapped`.

In [ ]:
from ch02.scripted import run_scripted_agent
from ch02.env import make_env
from ch02.viz import record_scripted_episode_video

env = make_env(
    obs_mode="state", control_mode="pd_joint_pos", render_mode=None,
)
rate = run_scripted_agent(env, n_episodes=10)
print(f"Scripted agent success rate: {rate:.0%}")

# Record one scripted rollout as MP4 (separate env in rgb_array mode).
video_env = make_env(
    obs_mode="state", control_mode="pd_joint_pos", render_mode="rgb_array",
)
record_scripted_episode_video(video_env, n_episodes=1, fps=20)


## 2.3 The LeRobot Dataset Standard

The chapter's expert data comes from `lerobot/svla_so101_pickplace` —
50 episodes of real-hardware teleoperated pick-and-place at 30 fps. Each
frame has a 6-DOF joint state, two USB-camera streams (`up` and `side`),
the recorded teleoperation action, and per-episode metadata.

### Listing 2.5: Loading the SO-101 pick-and-place expert dataset

In [ ]:
from lerobot.datasets import LeRobotDataset

dataset = LeRobotDataset(
    "lerobot/svla_so101_pickplace",
)
print(f"Total frames: {len(dataset)}")
print(f"Episodes: {dataset.num_episodes}")
print(f"Features: {list(dataset.features.keys())}")


In [ ]:
# Visualize: per-episode lengths across the 50 expert episodes
from ch02.viz import plot_episode_lengths
plot_episode_lengths(dataset);


### Listing 2.6: Inspecting a single frame and one episode

Inspect frame 0's tensor shapes and dtypes, then collect every frame that
belongs to episode 0 to confirm episode-length variation across the 50
trajectories.

In [ ]:
frame = dataset[0]
for key, val in frame.items():
    if hasattr(val, "shape"):
        print(f"  {key}: shape={val.shape}, dtype={val.dtype}")
    else:
        print(f"  {key}: {val}")

ep_indices = [i for i in range(len(dataset))
              if dataset[i]["episode_index"] == 0]
print(f"\nEpisode 0 length: {len(ep_indices)} steps")


In [ ]:
# Visualize: play episode 0 as MP4 (single up-camera preview)
from ch02.viz import record_dataset_episode_video
record_dataset_episode_video(dataset, episode_idx=0, cameras=("up",), fps=30)


## 2.4 Visualizing the Data

Three figures: keyframes from one expert episode, per-joint action
distributions comparing expert/scripted/random, and joint trajectories
overlaid across several expert episodes.

### Listing 2.7: Rendering expert keyframes from both camera views

A two-row filmstrip: `up` camera on top, `side` camera below, six
evenly-spaced keyframes from one expert episode. Live in `ch02.viz`;
shown here for transparency.

In [ ]:
from ch02.viz import record_dataset_episode_video

# Both camera views side-by-side, full episode at 30 fps (~10s clip).
record_dataset_episode_video(
    dataset, episode_idx=0, cameras=("up", "side"), fps=30,
)


### Listing 2.8: Per-joint action distributions

Collect actions from three sources and overlay per-dimension histograms.
Expert actions are pure dataset indexing (no env), so they're an inline
`np.stack`. The env-side helpers live in `ch02.viz`: `collect_actions(env,
None, ...)` samples uniformly, and `capture_scripted_actions` intercepts
`env.step` to record the joint commands the motion planner issues
mid-trajectory.

In [ ]:
import numpy as np
from ch02.viz import (
    collect_actions,
    capture_scripted_actions,
    plot_action_distributions,
)

expert = np.stack(
    [np.asarray(dataset[i]["action"]) for i in range(len(dataset))]
)
random_ = collect_actions(env, None, n_episodes=3)
scripted = capture_scripted_actions(env, n_episodes=3)

# Per-source min-max normalization handles the unit mismatch — expert
# is degrees, scripted is radians, random is normalized deltas. Without
# rescaling, random/scripted collapse into a single bar at the expert
# scale's origin and the comparison is unreadable.
plot_action_distributions(expert, scripted, random_)


### Figure 2.6: Per-joint trajectories across episodes

Overlay per-joint angle traces from the first five expert episodes
to show that episodes share coarse structure but diverge in timing —
a learned policy must be conditional on observations, not just a
memorized trajectory.

In [ ]:
from ch02.viz import plot_joint_trajectories

fig = plot_joint_trajectories(dataset, episode_indices=range(5))


## 2.5 The Data Pipeline

Three functions are the chapter's Ch3 export contract: `compute_stats`,
`normalize`/`denormalize`, and `make_pickplace_dataloader`. The first
two are type-along; the dataloader is a provided utility shown for
transparency.

### Listing 2.9: Computing normalization statistics manually

In [ ]:
import torch

def compute_stats(dataset):
    """Per-feature mean, std, min, max for state and action."""
    states, actions = [], []
    for i in range(len(dataset)):
        frame = dataset[i]
        states.append(frame["observation.state"])
        actions.append(frame["action"])
    states = torch.stack(states)
    actions = torch.stack(actions)
    return {
        "observation.state": {
            "mean": states.mean(0), "std": states.std(0),
            "min": states.min(0).values, "max": states.max(0).values,
        },
        "action": {
            "mean": actions.mean(0), "std": actions.std(0),
            "min": actions.min(0).values, "max": actions.max(0).values,
        },
    }

stats = compute_stats(dataset)
print({k: list(v.keys()) for k, v in stats.items()})


In [ ]:
# Visualize: per-dim mean ± std for action and state
from ch02.viz import plot_stats
plot_stats(stats, key="action");
plot_stats(stats, key="observation.state");


### Listing 2.10: Normalize and denormalize functions

The `(x, stats, key)` signature is the same for both functions so they
compose cleanly in the dataloader collate.

In [ ]:
def normalize(x, stats, key):
    """Z-score normalization: (x - mean) / std."""
    return (x - stats[key]["mean"]) / (stats[key]["std"] + 1e-8)

def denormalize(x, stats, key):
    """Inverse z-score normalization."""
    return x * (stats[key]["std"] + 1e-8) + stats[key]["mean"]

sample = dataset[0]["observation.state"]
normed = normalize(sample, stats, "observation.state")
recovered = denormalize(normed, stats, "observation.state")
assert torch.allclose(sample, recovered, atol=1e-5)
print("round-trip ok")


In [ ]:
# Visualize: normalization shifts a raw joint dim to ~zero mean, unit std
import numpy as np
from ch02.viz import JOINT_NAMES, plot_normalization_effect

# Sample 500 random frames to keep this cell fast (full sweep takes a few min)
sample_idx = np.random.RandomState(0).choice(len(dataset), 500, replace=False)
raw = np.array([
    float(dataset[int(i)]["observation.state"][0]) for i in sample_idx
])
mean_0 = float(stats["observation.state"]["mean"][0])
std_0 = float(stats["observation.state"]["std"][0])
normed = (raw - mean_0) / (std_0 + 1e-8)
plot_normalization_effect(raw, normed, dim_name=JOINT_NAMES[0]);


### Listing 2.11: Building the DataLoader (Ch3 export contract)

`make_pickplace_dataloader` is a provided utility from `ch02`. Its
signature is frozen — renaming or reordering breaks every downstream
chapter. The listing below shows the full implementation for
transparency; in practice you import it from the package.

In [ ]:
from ch02 import make_pickplace_dataloader

loader, stats = make_pickplace_dataloader(batch_size=4)
batch = next(iter(loader))
print(f"state mean per dim: {batch['observation.state'].mean(0)}")
print(f"image range: [{batch['observation.images.up'].min():.2f}, "
      f"{batch['observation.images.up'].max():.2f}]")
print(f"action shape: {batch['action'].shape}")


In [ ]:
# Visualize: one batch sample — image (normalized to [0,1]) + state vector
import matplotlib.pyplot as plt
fig, (ax_img, ax_state) = plt.subplots(1, 2, figsize=(12, 4))
img = batch["observation.images.up"][0].permute(1, 2, 0).numpy()
ax_img.imshow(img); ax_img.axis("off")
ax_img.set_title("batch[0] observation.images.up (in [0,1])")
state = batch["observation.state"][0].numpy()
ax_state.bar(range(len(state)), state, color="tab:blue", alpha=0.7)
ax_state.axhline(0, color="grey", linewidth=0.8)
ax_state.set_title("batch[0] observation.state (z-score normalized)")
ax_state.set_xlabel("joint index")
plt.tight_layout();


## 2.6 Summary

What this chapter delivers, in one screen:

- **`PickCubeSO100-v1`** is the carrier task — 6-DOF SO-100 arm, single cube, fixed start, fixed target. Served by ManiSkill3 over SAPIEN. Same embodiment, same task family, same observation/action shapes through Chapter 9.
- **The Gymnasium API** is the universal interface — `reset()` returns an observation, `step(action)` returns the next observation + reward + termination flags. Sim and real-hardware envs in later chapters expose the same interface.
- **A random agent never solves it.** A motion-planner-backed scripted policy raises the success rate but plateaus below expert teleoperation — it can't recover from misalignment or adapt to contact dynamics.
- **Expert demonstrations** come from `lerobot/svla_so101_pickplace` (50 episodes, 30 fps, two camera streams, the natural-language task description). Loaded via `lerobot.datasets.LeRobotDataset`.
- **Action distributions** for expert / scripted / random reveal the structural gap a learned policy must close — multi-modal expert clusters vs. lower-variance scripted commands vs. uniform random.
- **Normalization** is z-score for state and action; images stay as `float32` in `[0, 1]` (clamped defensively). `denormalize` recovers environment-scale predictions before `env.step()`.

### The Chapter 3 contract

Chapter 3 imports these directly from `ch02`:

```python
from ch02 import make_pickplace_dataloader, normalize, denormalize, StatsDict
```

`make_pickplace_dataloader(dataset_id, batch_size, shuffle)` returns `(DataLoader, stats)`. Argument order is frozen — Chapter 3 and every chapter after it depends on this signature.

In [ ]:
# Smoke check: the Ch3 contract is importable + shaped correctly.
from ch02 import (
    StatsDict, denormalize, make_pickplace_dataloader, normalize,
)
import inspect

sig = inspect.signature(make_pickplace_dataloader)
print("make_pickplace_dataloader signature:")
print(f"  params: {list(sig.parameters.keys())}")
print(f"  return: {sig.return_annotation}")
print()
print("Frozen Ch3 imports:")
for name, obj in [
    ("make_pickplace_dataloader", make_pickplace_dataloader),
    ("normalize", normalize),
    ("denormalize", denormalize),
    ("StatsDict", StatsDict),
]:
    print(f"  ch02.{name}: {obj!r}")
